# First-contact emitter assignment with the emitter-aware ontology

This notebook ingests the first `PY_CONTACT_LOG` line from `LuaHistory_2026-06-23.txt`, extracts CMO observation features, retrieves platform/operator hypotheses from the new KG ontology, applies the probability layer, and builds an LLM explanation payload.

In [17]:
from pathlib import Path
import os, sys, json, math
from dataclasses import asdict

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'combat_id_calibration').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

LUA_HISTORY = REPO_ROOT / 'LuaHistory_2026-06-23.txt'
WORK_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'first_contact_emitter_assignment'
WORK_DIR.mkdir(parents=True, exist_ok=True)

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = 'password123' #os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None

In [18]:
# Diagnostic: list every relationship type currently present in the Neo4j graph.
# Run this before hypothesis generation to decide which relationship types are safe
# to include in a restricted emitter-to-platform traversal.
from neo4j import GraphDatabase

relationship_type_query = """
MATCH ()-[rel]->()
RETURN type(rel) AS relationship_type, count(*) AS relationship_count
ORDER BY relationship_count DESC, relationship_type ASC
"""

if not NEO4J_PASSWORD:
    print('Set NEO4J_PASSWORD to list Neo4j relationship types')
else:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    try:
        with driver.session(**({'database': NEO4J_DATABASE} if NEO4J_DATABASE else {})) as session:
            relationship_type_rows = [dict(row) for row in session.run(relationship_type_query)]
    finally:
        driver.close()

    try:
        import pandas as pd
        display(pd.DataFrame(relationship_type_rows))
    except ImportError:
        for row in relationship_type_rows:
            print(f"{row['relationship_type']}: {row['relationship_count']}")


,relationship_type,relationship_count
0,MENTIONED_IN,2872
1,FACT,2250
2,OPERATED_BY,176
3,HAS_SENSOR,122
4,HAS_VARIANT,90
5,HAS_SUBSYSTEM,83
6,OPERATOR_COUNTRY,70
7,BASED_AT,59
8,IS_A,58
9,ALSO_KNOWN_AS,56


In [19]:
from combat_id_calibration.cmo_observation_ingest import parse_observation_line, write_observations_jsonl, populate_observations_neo4j
first_line = next(line for line in LUA_HISTORY.read_text(encoding='utf-8-sig', errors='replace').splitlines() if line.startswith('PY_CONTACT_LOG'))
obs = parse_observation_line(first_line, source_line=1)
write_observations_jsonl([obs], WORK_DIR / 'first_contact_observation.jsonl')
asdict(obs)

{'observation_id': '13e4329885a9f92e',
 'time': 1844772240,
 'sensor_aircraft': 'Typhoon FGR.4',
 'emission_sensor_name': 'Slot Back [N-010 Zhuk-M]',
 'emission_age': 33.900054931641,
 'emission_solid': True,
 'emission_type': 2001,
 'emission_role': 2122,
 'emission_latitude': 44.640933862914,
 'emission_longitude': 31.890743606263,
 'emission_heading': 331.40036010742,
 'emission_altitude': 10316.349609375,
 'emission_speed': 479.64691162109,
 'emission_target_type': 'Type: Multirole (Fighter/Attack)',
 'emission_classificationlevel': 2,
 'source_line': 1,
 'source': 'cmo_lua',
 'schema': 'cmo_emission_observation_v1'}

In [34]:
obs

EmissionObservation(observation_id='13e4329885a9f92e', time=1844772240, sensor_aircraft='Typhoon FGR.4', emission_sensor_name='Slot Back [N-010 Zhuk-M]', emission_age=33.900054931641, emission_solid=True, emission_type=2001, emission_role=2122, emission_latitude=44.640933862914, emission_longitude=31.890743606263, emission_heading=331.40036010742, emission_altitude=10316.349609375, emission_speed=479.64691162109, emission_target_type='Type: Multirole (Fighter/Attack)', emission_classificationlevel=2, source_line=1, source='cmo_lua', schema='cmo_emission_observation_v1')

In [31]:
rubin = [line for line in LUA_HISTORY.read_text(encoding='utf-8-sig', errors='replace').splitlines() if line.startswith('PY_CONTACT_LOG')][7]
rubin

'PY_CONTACT_LOG  Time : 1844772420 , Sensor_aircraft : MiG-29KUB Fulcrum D , Emission_sensor_name : Slot Back [N-019 Rubin] , Emission_age : 31.900085449219 , Emission_solid : false , Emission_type : 2001 , Emission_role : 2113 , Emission_latitude : 45.568013757942 , Emission_longitude : 30.823081443229 , Emission_heading : 170.09051513672 , Emission_altitude : 10242.206054688 , Emission_speed : 454.21356201172 , Emission_target_type : Type: Unknown air contact , Emission_classificationlevel : 1'

In [33]:
obs_rubin = parse_observation_line(rubin)
obs_rubin

EmissionObservation(observation_id='8ec0578d33ab0134', time=1844772420, sensor_aircraft='MiG-29KUB Fulcrum D', emission_sensor_name='Slot Back [N-019 Rubin]', emission_age=31.900085449219, emission_solid=False, emission_type=2001, emission_role=2113, emission_latitude=45.568013757942, emission_longitude=30.823081443229, emission_heading=170.09051513672, emission_altitude=10242.206054688, emission_speed=454.21356201172, emission_target_type='Type: Unknown air contact', emission_classificationlevel=1, source_line=None, source='cmo_lua', schema='cmo_emission_observation_v1')

In [20]:
first_line

'PY_CONTACT_LOG  Time : 1844772240 , Sensor_aircraft : Typhoon FGR.4 , Emission_sensor_name : Slot Back [N-010 Zhuk-M] , Emission_age : 33.900054931641 , Emission_solid : true , Emission_type : 2001 , Emission_role : 2122 , Emission_latitude : 44.640933862914 , Emission_longitude : 31.890743606263 , Emission_heading : 331.40036010742 , Emission_altitude : 10316.349609375 , Emission_speed : 479.64691162109 , Emission_target_type : Type: Multirole (Fighter/Attack) , Emission_classificationlevel : 2'

In [21]:
# Optional: write the dynamic observation into the same Neo4j graph.
if NEO4J_PASSWORD:
    populate_observations_neo4j([obs], NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
    print('First observation ingested into Neo4j')
else:
    print('Set NEO4J_PASSWORD to ingest the observation into Neo4j')

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (obs, contact) { ... }', position=<SummaryInputPosition line=22, column=9, offset=1391>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1391, 'line': 22, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        MERGE (obs:Observation {id: $observation_id})\n          SET obs.schema = $schema, obs.time = $time, obs.source = $source,\n              obs.source_line = $source_line, obs.age_seconds = $emission_age,\n              obs.solid = $emission_solid, obs.latitude = $emission_latitude,\n              obs.longitude = $emi

First observation ingested into Neo4j


In [35]:
if NEO4J_PASSWORD:
    populate_observations_neo4j([obs_rubin], NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
    print('First observation ingested into Neo4j')
else:
    print('Set NEO4J_PASSWORD to ingest the observation into Neo4j')

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (obs, contact) { ... }', position=<SummaryInputPosition line=22, column=9, offset=1391>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1391, 'line': 22, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        MERGE (obs:Observation {id: $observation_id})\n          SET obs.schema = $schema, obs.time = $time, obs.source = $source,\n              obs.source_line = $source_line, obs.age_seconds = $emission_age,\n              obs.solid = $emission_solid, obs.latitude = $emission_latitude,\n              obs.longitude = $emi

First observation ingested into Neo4j


In [36]:
from combat_id_calibration.hypothesis_generation import fetch_graph_hypotheses, select_offline_hypotheses, emitter_aliases

# seed_candidates = [
#     {'hypothesis':'MiG-29 Fulcrum C', 'operator_nation':'Ukraine', 'emitter_aliases':['Slot Back [N-010 Zhuk-M]','N-010 Zhuk-M','Zhuk-M'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,800], 'typical_altitude_m':[0,18000], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
#     {'hypothesis':'MiG-29SMT', 'operator_nation':'Russia', 'emitter_aliases':['N-010 Zhuk-M','Zhuk-M'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,800], 'typical_altitude_m':[0,18000], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
#     {'hypothesis':'MiG-35', 'operator_nation':'Russia', 'emitter_aliases':['Zhuk-M','Zhuk-AE'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,900], 'typical_altitude_m':[0,17500], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
# ]
if NEO4J_PASSWORD:
    kg_rows = fetch_graph_hypotheses(obs, 100, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE, relationship_types=["HAS_SENSOR", "HAS_PLATFORM", "VARIANT_OF", "HAS_VARIANT", "OPERATED_BY", "OPERATOR_COUNTRY", "USES_RADAR", "ALSO_KNOWN_AS"]) #or seed_candidates
else:
    print('Neo4j unavailable')
hypotheses = select_offline_hypotheses(obs, kg_rows, 25)
hypotheses

emitter_aliases: ['Slot Back [N-010 Zhuk-M]', 'N-010 Zhuk-M', 'Slot Back', 'N010 Zhuk M', 'N010 Zhuk', 'Zhuk M', 'Slot Back N010 Zhuk M', 'Slot Back N010 Zhuk']


[{'hypothesis': 'MiG-29K',
  'operator_nation': 'India (Indian Navy)',
  'aircraft_variant': 'MiG-29K',
  'emitter_variant': 'N010 Zhuk-AE',
  'emitter_aliases': ['N010 Zhuk-AE', 'Zhuk-ME'],
  'platform_class': 'Type: Multirole (Fighter/Attack)',
  'typical_speed_kt': [0.0, nan],
  'typical_altitude_m': [0.0, nan],
  'kg_support_count': 13,
  'evidence_paths': [['HAS_VARIANT', 'VARIANT_OF', 'USES_RADAR', 'VARIANT_OF'],
   ['OPERATOR_COUNTRY', 'OPERATOR_COUNTRY', 'USES_RADAR', 'VARIANT_OF'],
   ['OPERATED_BY', 'OPERATED_BY', 'USES_RADAR', 'VARIANT_OF'],
   ['HAS_VARIANT', 'HAS_VARIANT', 'USES_RADAR', 'VARIANT_OF'],
   ['HAS_SENSOR', 'HAS_SENSOR', 'USES_RADAR', 'VARIANT_OF'],
   ['HAS_VARIANT', 'VARIANT_OF', 'USES_RADAR'],
   ['OPERATOR_COUNTRY', 'OPERATOR_COUNTRY', 'USES_RADAR'],
   ['OPERATED_BY', 'OPERATED_BY', 'OPERATOR_COUNTRY', 'USES_RADAR'],
   ['OPERATED_BY', 'OPERATED_BY', 'USES_RADAR'],
   ['OPERATOR_COUNTRY', 'OPERATED_BY', 'OPERATED_BY', 'USES_RADAR'],
   ['HAS_SENSOR', 'HAS_

In [37]:
if NEO4J_PASSWORD:
    kg_rows = fetch_graph_hypotheses(obs_rubin, 100, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE, relationship_types=["HAS_SENSOR", "HAS_PLATFORM", "VARIANT_OF", "HAS_VARIANT", "OPERATED_BY", "OPERATOR_COUNTRY", "USES_RADAR", "ALSO_KNOWN_AS"]) #or seed_candidates
else:
    print('Neo4j unavailable')
hypotheses_rubin = select_offline_hypotheses(obs_rubin, kg_rows, 25)
hypotheses_rubin

emitter_aliases: ['Slot Back [N-019 Rubin]', 'N-019 Rubin', 'Slot Back', 'N019 Rubin', 'Rubin', 'Slot Back N019 Rubin']


[{'hypothesis': 'MiG-29A variant delivery context',
  'operator_nation': 'Unknown',
  'aircraft_variant': 'MiG-29A variant delivery context',
  'emitter_variant': 'N019 Rubin radar',
  'emitter_aliases': ['N019 Rubin radar'],
  'platform_class': 'Type: Unknown air contact',
  'typical_speed_kt': [0.0, nan],
  'typical_altitude_m': [0.0, nan],
  'kg_support_count': 1,
  'evidence_paths': [['HAS_SENSOR']],
  'semantic_match_score': 22.0},
 {'hypothesis': 'MiG-29 (Product 9.12)',
  'operator_nation': 'Unknown',
  'aircraft_variant': 'MiG-29 (Product 9.12)',
  'emitter_variant': 'Phazotron N019 Rubin radar',
  'emitter_aliases': ['Phazotron N019 Rubin radar'],
  'platform_class': 'Type: Unknown air contact',
  'typical_speed_kt': [0.0, nan],
  'typical_altitude_m': [0.0, nan],
  'kg_support_count': 1,
  'evidence_paths': [['HAS_SENSOR']],
  'semantic_match_score': 22.0}]

In [38]:
kg_rows

[{'hypothesis': 'MiG-29 (Product 9.12)',
  'operator_nation': 'Unknown',
  'aircraft_variant': 'MiG-29 (Product 9.12)',
  'emitter_variant': 'Phazotron N019 Rubin radar',
  'emitter_aliases': ['Phazotron N019 Rubin radar'],
  'platform_class': 'Type: Unknown air contact',
  'typical_speed_kt': [0.0, nan],
  'typical_altitude_m': [0.0, nan],
  'kg_support_count': 1,
  'evidence_paths': [['HAS_SENSOR']],
  'semantic_match_score': 22.0},
 {'hypothesis': 'MiG-29A variant delivery context',
  'operator_nation': 'Unknown',
  'aircraft_variant': 'MiG-29A variant delivery context',
  'emitter_variant': 'N019 Rubin radar',
  'emitter_aliases': ['N019 Rubin radar'],
  'platform_class': 'Type: Unknown air contact',
  'typical_speed_kt': [0.0, nan],
  'typical_altitude_m': [0.0, nan],
  'kg_support_count': 1,
  'evidence_paths': [['HAS_SENSOR']],
  'semantic_match_score': 22.0}]

In [39]:
from combat_id_calibration.probability_model import platform_operator_nation_distribution
from combat_id_calibration.hypothesis_generation import evidence_paths_query_id

SCENARIO_ID = 'LuaHistory_2026-06-23_first_contact'

def hypothesis_feature_row(obs, h):
    alias_match = any(a.lower() in obs.emission_sensor_name.lower() for a in h.get('emitter_aliases', []))
    speed_low, speed_high = h.get('typical_speed_kt', [0, 2500])
    alt_low, alt_high = h.get('typical_altitude_m', [0, 25000])
    speed_match = speed_low <= obs.emission_speed <= speed_high
    altitude_match = alt_low <= obs.emission_altitude <= alt_high
    support_count = float(h.get('kg_support_count', len(h.get('evidence_paths', [])) or 0))
    # These feature columns are the contract consumed by probability_model.py.
    return {
        'scenario_id': SCENARIO_ID,
        'contact_id': obs.observation_id,
        'observation_time': obs.time,
        'hypothesis': h['hypothesis'],
        'operator_nation': h.get('operator_nation', 'unknown'),
        'supporting_path_count': support_count,
        'contradicting_path_count': float(h.get('contradicting_path_count', 0)),
        'mean_source_reliability': float(h.get('mean_source_reliability', 0.75)),
        'recency': float(h.get('recency', 1.0)),
        'shortest_path_to_platform_class': float(h.get('shortest_path_to_platform_class', 1 if h.get('platform_class') else 0)),
        'emission_match_score': 1.0 if alias_match else 0.0,
        'kinematic_match_score': (float(speed_match) + float(altitude_match)) / 2.0,
        'contradiction_score': float(h.get('contradiction_score', 0.0)),
        'evidence_query_id': h.get('evidence_query_id') or evidence_paths_query_id(h.get('evidence_paths', [])),
        'features': {
            'emitter_alias_match': alias_match,
            'observed_speed_kt': obs.emission_speed,
            'observed_altitude_m': obs.emission_altitude,
            'observed_latitude': obs.emission_latitude,
            'observed_longitude': obs.emission_longitude,
        },
    }

feature_rows = [hypothesis_feature_row(obs, h) for h in hypotheses]
probability_record = platform_operator_nation_distribution(feature_rows)
assignments = [
    {
        **hypotheses[candidate['candidate_index']],
        'probability': candidate['probability'],
        'operator_nation': candidate['operator_nation'],
        'logit': candidate['logit'],
        'evidence_query_id': candidate['evidence_query_id'],
        'features': feature_rows[candidate['candidate_index']]['features'],
    }
    for candidate in probability_record['candidates']
]
(WORK_DIR / 'first_contact_feature_rows.jsonl').write_text(''.join(json.dumps(row, sort_keys=True)+'\n' for row in feature_rows), encoding='utf-8')
(WORK_DIR / 'first_contact_probability_assignment.jsonl').write_text(json.dumps(probability_record, sort_keys=True)+'\n', encoding='utf-8')
probability_record



operator_nations : ['India (Indian Navy)', 'India (Indian Navy)', 'India (Indian Navy)', 'India (Indian Navy)', 'India (Indian Navy)', 'India (Indian Navy)', 'India', 'India', 'India', 'India', 'India', 'India', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'India']
distance_evidence : [None, None, None, None, None, None, {'emitter_latitude': 44.640933862914, 'emitter_longitude': 31.890743606263, 'operator_nation_latitude': 35.67, 'operator_nation_longitude': 68.11, 'operator_nation_distance_reference': 'border', 'operator_nation_distance_km': 3205.5571601250326, 'operator_nation_distance_score': 0.20133631100547297, 'operator_nation_distance_logit_adjustment': 0.20133631100547297}, {'emitter_latitude': 44.640933862914, 'emitter_longitude': 31.890743606263, 'operator_nation_latitude': 35.67, 'operator_nation_longitude': 68.11, 'operator_nation_distance_reference': 'border', 'operator_nation_distance_km': 3205.557

{'schema': 'platform_operator_nation_probability_v1',
 'scenario_id': 'LuaHistory_2026-06-23_first_contact',
 'contact_id': '13e4329885a9f92e',
 'observation_time': '1844772240',
 'top_platform': 'Mikoyan MiG-29K',
 'top_platform_probability': 0.5283124378879662,
 'top_operator_nation': 'Russia',
 'top_operator_nation_probability': 0.5057334479773969,
 'platform_probabilities': {'Mikoyan MiG-29K': 0.5283124378879662,
  'MiG-29K': 0.47168756211203366},
 'operator_nation_probabilities': {'Russia': 0.5057334479773969,
  'India': 0.2820848522200989,
  'India (Indian Navy)': 0.2121816998025041},
 'candidates': [{'candidate_index': 6,
   'platform': 'MiG-29K',
   'operator_nation': 'India',
   'probability': 0.04325097705158827,
   'logit': 9.551336311005475,
   'base_logit': 9.350000000000001,
   'evidence_query_id': 'HAS_VARIANT>VARIANT_OF>USES_RADAR>VARIANT_OF|OPERATOR_COUNTRY>OPERATOR_COUNTRY>USES_RADAR>VARIANT_OF|OPERATED_BY>OPERATED_BY>USES_RADAR>VARIANT_OF|HAS_VARIANT>HAS_VARIANT>USES

In [40]:
from combat_id_calibration.hypothesis_generation import build_llm_hypothesis_prompt
from combat_id_calibration.llm_explainer import build_explanation_payload

evidence = {
    'supporting_evidence': [
        {
            'text': f"{row['hypothesis']} has emitter_alias_match={row['features']['emitter_alias_match']} and kinematic_match_score={row['kinematic_match_score']:.2f}",
            'source': row.get('evidence_query_id') or row['hypothesis'],
        }
        for row in feature_rows
    ],
    'missing_evidence': [
        'Collect additional emitter scans, track kinematics, IFF, location context, and source reliability before treating this as definitive.'
    ],
}
explanation_payload = build_explanation_payload(probability_record, evidence)
explanation_payload.update({
    'observation': asdict(obs),
    'emitter_aliases': emitter_aliases(obs.emission_sensor_name),
    'probability_assignments': assignments,
    'llm_instruction': 'Explain why the probability model favored these identities/operators. Do not change probabilities.',
    'hypothesis_prompt_context': build_llm_hypothesis_prompt(obs, hypotheses, len(hypotheses)),
})
(WORK_DIR / 'first_contact_explanation_payload.json').write_text(json.dumps(explanation_payload, indent=2, sort_keys=True), encoding='utf-8')
explanation_payload



{'schema': 'llm_explainer_payload_v1',
 'scenario_id': 'LuaHistory_2026-06-23_first_contact',
 'contact_id': '13e4329885a9f92e',
 'observation_time': '1844772240',
 'summary': 'Most likely emitter platform is Mikoyan MiG-29K with probability 0.528; most likely operator nation is Russia with probability 0.506.',
 'confidence_limits': ['Top platform probability is below 0.60, so the assignment should be treated as tentative.',
  'The two leading platform hypotheses are separated by less than 0.15 probability.'],
 'supporting_evidence': [{'text': 'MiG-29K has emitter_alias_match=False and kinematic_match_score=0.00',
   'source': 'HAS_VARIANT>VARIANT_OF>USES_RADAR>VARIANT_OF|OPERATOR_COUNTRY>OPERATOR_COUNTRY>USES_RADAR>VARIANT_OF|OPERATED_BY>OPERATED_BY>USES_RADAR>VARIANT_OF|HAS_VARIANT>HAS_VARIANT>USES_RADAR>VARIANT_OF|HAS_SENSOR>HAS_SENSOR>USES_RADAR>VARIANT_OF|HAS_VARIANT>VARIANT_OF>USES_RADAR|OPERATOR_COUNTRY>OPERATOR_COUNTRY>USES_RADAR|OPERATED_BY>OPERATED_BY>OPERATOR_COUNTRY>USES_RA

In [ ]:
import re
import urllib.error

from combat_id_calibration.graph_ingest import DEFAULT_MODEL, DEFAULT_OLLAMA_URL, ollama_generate

def _extract_json_object(text):
    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

def explain_with_llm(explanation_payload, model=None, ollama_url=None):
    """Send the explanation payload to a local Ollama LLM and return text plus model probability.

    The LLM is downstream of probability_model.py: it explains the supplied
    probability, but never computes or edits the probability. Ollama must be
    running locally with the selected model available. If the local Ollama
    endpoint is unavailable, the deterministic payload summary is returned so
    the notebook remains runnable offline without remote LLM resources.
    """
    top_probability = float(probability_record['top_platform_probability'])
    selected_model = model or os.getenv('OLLAMA_MODEL', DEFAULT_MODEL)
    selected_ollama_url = ollama_url or os.getenv('OLLAMA_URL', DEFAULT_OLLAMA_URL)
    prompt = """You explain calibrated combat-identification model outputs.
Do not alter, recalculate, round aggressively, or invent probabilities.
Return JSON with keys explanation and probability.

Payload:
""" + json.dumps(explanation_payload, indent=2, sort_keys=True)

    try:
        raw_text = ollama_generate(prompt, model=selected_model, ollama_url=selected_ollama_url)
    except (OSError, urllib.error.URLError, TimeoutError) as exc:
        return {
            'probability': top_probability,
            'explanation': explanation_payload['summary'],
            'llm_raw_output': None,
            'ollama_model': selected_model,
            'ollama_url': selected_ollama_url,
            'note': f'Local Ollama endpoint unavailable ({exc}); returned deterministic payload summary instead.',
        }

    parsed = _extract_json_object(raw_text) or {}
    return {
        'probability': top_probability,
        'explanation': parsed.get('explanation', raw_text or explanation_payload['summary']),
        'llm_raw_output': raw_text,
        'ollama_model': selected_model,
        'ollama_url': selected_ollama_url,
    }

llm_explanation = explain_with_llm(explanation_payload)
(WORK_DIR / 'first_contact_llm_explanation.json').write_text(json.dumps(llm_explanation, indent=2, sort_keys=True), encoding='utf-8')
llm_explanation

